In [4]:
from mlflow import MlflowClient

In [5]:
client = MlflowClient()
client.create_registered_model("Insurance")

2026/02/06 22:19:16 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.schemas
2026/02/06 22:19:16 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.tables
2026/02/06 22:19:16 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.types
2026/02/06 22:19:16 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.constraints
2026/02/06 22:19:16 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.defaults
2026/02/06 22:19:16 INFO alembic.runtime.plugins: setup plugin alembic.autogenerate.comments
2026/02/06 22:19:17 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2026/02/06 22:19:17 INFO alembic.runtime.migration: Will assume non-transactional DDL.


<RegisteredModel: aliases={}, creation_timestamp=1770445157226, deployment_job_id=None, deployment_job_state=None, description=None, last_updated_timestamp=1770445157226, latest_versions=[], name='Insurance', tags={}>

In [ ]:
# Insurance filter string
insurance_filter_string = "name LIKE 'Insurance%'"

# Search for Insurance models
print(client.search_registered_models(filter_string=insurance_filter_string))

# Not Insurance filter string
not_insurance_filter_string = "name != 'Insurance'"

# Search for non Insurance models
print(client.search_registered_models(filter_string=not_insurance_filter_string))

[<RegisteredModel: aliases={}, creation_timestamp=1770445157226, deployment_job_id=None, deployment_job_state=None, description=None, last_updated_timestamp=1770445157226, latest_versions=[], name='Insurance', tags={}>]
[]


: 

### Example code to register models
```python
# Register the first (2022) model
mlflow.register_model("model_2022", "Insurance")

# Register the second (2023) model
mlflow.register_model(f"runs:/{run_id}/model_2023", "Insurance")
```

### Registering from training and Searching:

```python
# Log the model using scikit-learn flavor
mlflow.sklearn.log_model(lr, "model", registered_model_name="Insurance")
insurance_filter_string = "name = 'Insurance'"

# Search for Insurance models
print(client.search_registered_models(filter_string=insurance_filter_string))
```

```python
# Transition version 2 of Insurance model to stable stage
client.transition_model_version_stage(name="Insurance", version=2,
        stage="Production"
    )
```


```python
# Transition version 1 of Insurance model to archive stage
client.transition_model_version_stage(name="Insurance", version=1,
        stage="Archived"
    )
```

### Loading the Insurance model of the Production Stage version

```python
# Load the Production stage of Insurance model using scikit-learn flavor
model = mlflow.sklearn.load_model("models:/Insurance/Production")

# Run prediction on our test data
model.predict(X_test)
```

### Running the mlflow.projects with parameters:
```python
import mlflow

# Set the run function from the MLflow Projects module
mlflow.projects.run(
    uri='./',
    entry_point='main',
    experiment_name='Insurance',
  	env_manager='local',
  	# Set parameters for n_jobs and fit_intercept
  	parameters={
        'n_jobs_param': 2, 
        'fit_intercept_param': False
    }
)
```

### Model Engineering and Model Evalution connected workflow:
```python
# Set run method to model_engineering
model_engineering = mlflow.projects.run(
    uri='./',
    # Set entry point to model_engineering
    entry_point='model_engineering',
    experiment_name='Insurance',
    # Set the parameters for n_jobs and fit_intercept
    parameters={
        'n_jobs_param': 2, 
        'fit_intercept_param': False
    },
    env_manager='local'
)

# Set Run ID of model training to be passed to Model Evaluation step
model_engineering_run_id = model_engineering.run_id
print(model_engineering_run_id)

# Set the MLflow Projects run method
model_evaluation = mlflow.projects.run(
    uri="./",
    # Set the entry point to model_evaluation
    entry_point="model_evaluation",
  	# Set the parameter run_id to the run_id output of previous step
    parameters={
        "run_id": model_engineering_run_id,
    },
    env_manager="local"
)

print(model_evaluation.get_status())
```
